In [7]:
# --- Cell 1: Notebook Header, Logging Configuration, and Library Imports ---

"""
Notebook: 01_ingest_earthquake_data.ipynb

Purpose:
This script manages the initial ingestion phase of raw earthquake data.
It connects to the USGS API to extract seismic information and defines the
necessary data schema for subsequent processing. The objective is to prepare
the data for storage in the Bronze layer of the Lakehouse architecture,
serving as a raw, untransformed data source.

Author: Carlos Diaz Hernandez
Date: 2025-03-06 (YYYY-MM-DD format for standardization)
"""

# Configures a basic logging system for effective monitoring in production.
# This allows capturing informational messages, warnings, and errors throughout the script's execution.
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Standard Python libraries
import json
from datetime import datetime, timedelta, timezone # Added timezone for explicit UTC

# Third-party libraries for data handling and HTTP requests
import requests
import pandas as pd

# PySpark libraries for distributed processing and DataFrame manipulation.
# SparkSession: The entry point for programming Spark with the DataFrame and SQL API.
# StructType, StructField: Used to define the schema for Spark DataFrames.
# Spark data types: Ensures correct data interpretation and type consistency.
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType, TimestampType

logger.info("All necessary libraries have been imported successfully.")


StatementMeta(, 2d4ce2be-7e40-44ad-9bfc-5c2e276d3529, 9, Finished, Available, Finished)

2025-06-05 05:04:34,212 - INFO - All necessary libraries have been imported successfully.


In [8]:
# --- Cell 2: Data Schema Definition ---

# Defines the data schema for earthquake records obtained from the USGS API,
# along with additional metadata for auditing. This schema is critical for
# ensuring type consistency and data quality when loading data into a Spark
# DataFrame and subsequently into the Lakehouse's Bronze layer.
earthquake_schema = StructType([
    StructField("id", StringType(), True),           # Unique event ID from USGS
    StructField("mag", DoubleType(), True),          # Magnitude of the earthquake
    StructField("place", StringType(), True),        # Geographical location of the earthquake
    StructField("time", LongType(), True),           # Time of the event in milliseconds since the Unix epoch
    StructField("updated", LongType(), True),        # Time of the last update for the event (milliseconds epoch)
    StructField("tz", StringType(), True),           # Timezone offset from UTC in minutes
    StructField("url", StringType(), True),          # URL to the API for more event details
    StructField("detail", StringType(), True),       # URL for additional event details
    StructField("felt", LongType(), True),           # Number of reports from people who felt the earthquake
    StructField("cdi", DoubleType(), True),          # Community Decimal Intensity (CDI)
    StructField("mmi", DoubleType(), True),          # Modified Mercalli Intensity (MMI)
    StructField("alert", StringType(), True),        # USGS alert level (green, yellow, orange, red)
    StructField("status", StringType(), True),       # Status of the event (automatic, reviewed, deleted)
    StructField("tsunami", LongType(), True),        # Tsunami warning indicator (1 if warning issued, 0 if not)
    StructField("sig", LongType(), True),            # Significance or impact of the event
    StructField("net", StringType(), True),          # Seismic network that reported the event
    StructField("code", StringType(), True),         # Event identification code on network
    StructField("ids", StringType(), True),          # Comma-separated list of related event IDs
    StructField("sources", StringType(), True),      # Comma-separated list of data sources
    StructField("types", StringType(), True),        # Comma-separated list of event types
    StructField("nst", LongType(), True),            # Number of monitoring stations used for location
    StructField("dmin", DoubleType(), True),         # Horizontal distance from epicenter to the nearest station (degrees)
    StructField("rms", DoubleType(), True),          # Root-mean-square of the travel time residuals (seconds)
    StructField("gap", DoubleType(), True),          # Largest azimuthal gap in degrees between stations
    StructField("magType", StringType(), True),      # Type of magnitude reported (e.g., mb, ml, Mww)
    StructField("type", StringType(), True),         # Primary classification of the event (e.g., earthquake, quarry blast)
    StructField("title", StringType(), True),        # Descriptive title of the event
    StructField("longitude", DoubleType(), True),    # Longitude coordinate of the epicenter
    StructField("latitude", DoubleType(), True),     # Latitude coordinate of the epicenter
    StructField("depth", DoubleType(), True),        # Depth of the event in kilometers
    StructField("ingestion_timestamp_utc", TimestampType(), False) # Timestamp of when the record was ingested (UTC)
])

logger.info("'earthquake_schema' data schema defined and ready for use, including 'id' and 'ingestion_timestamp_utc'.")


StatementMeta(, 2d4ce2be-7e40-44ad-9bfc-5c2e276d3529, 10, Finished, Available, Finished)

2025-06-05 05:04:48,796 - INFO - 'earthquake_schema' data schema defined and ready for use, including 'id' and 'ingestion_timestamp_utc'.


In [9]:
# --- Cell 3: Configuration Parameters ---

# This section defines all key parameters for the data ingestion process.
# Centralizing these values ensures easy modification and maintainability,
# supporting environmental configuration if needed in future iterations.

# USGS API Endpoint:
# Base URL for querying earthquake data from the United States Geological Survey.
USGS_API_BASE_URL = "https://earthquake.usgs.gov/fdsnws/event/1/query"

# Data Fetching Window:
# Specifies the number of days prior to the current ingestion timestamp
# for which earthquake data will be retrieved. A longer window fetches more historical data.
DAYS_TO_FETCH = 365 # Configured to fetch a full year of historical data.

# Minimum Magnitude Filter:
# Sets the lower threshold for earthquake magnitudes to be included in the ingestion.
# Adjusting this value directly impacts the volume and granularity of the ingested data.
# A value of 2.5 is chosen to ensure a sufficient dataset for analysis, including
# less significant, but still relevant, seismic events.
MIN_MAGNITUDE = 2.5

# Ingestion Timestamp:
# Captures the exact UTC timestamp when this particular ingestion run initiated.
# Using UTC for consistency across different environments and time zones.
# This value is crucial for auditing, data lineage, and can be used for partitioning
# the raw data in the Lakehouse (e.g., year, month, day of ingestion).
INGESTION_DATETIME = datetime.now(timezone.utc) # Explicitly set to UTC

# Lakehouse Bronze Layer Target:
# Defines the fully qualified name of the table in the Bronze layer of the Lakehouse
# where the raw, untransformed data will be stored. This serves as the immutable
# landing zone for all ingested earthquake records.
BRONZE_TABLE_NAME = "bronze_usgs_earthquakes"

# Log the configured parameters for traceability and debugging in production environments.
logger.info("Ingestion Configuration Loaded:")
logger.info(f"  USGS API Base URL: {USGS_API_BASE_URL}")
logger.info(f"  Data Fetching Window: Last {DAYS_TO_FETCH} days")
logger.info(f"  Minimum Magnitude: {MIN_MAGNITUDE}")
logger.info(f"  Ingestion Datetime (UTC): {INGESTION_DATETIME.isoformat()}") # Using ISO format for clear timestamp
logger.info(f"  Bronze Target Table: {BRONZE_TABLE_NAME}")

# Spark Session Initialization (Commented for Fabric environments):
# In Azure Fabric notebooks, the 'spark' variable is typically pre-initialized and available globally.
# If this were a standalone Python script, explicit creation would be necessary.
# try:
#     spark = SparkSession.builder \
#                         .appName("USGSEarthquakeIngestion") \
#                         .getOrCreate()
#     logger.info("Spark Session initialized or retrieved successfully.")
# except Exception as e:
#     logger.error(f"Error initializing or retrieving Spark Session: {e}")
#     raise # Re-raise the exception to halt execution if Spark cannot be started.


StatementMeta(, 2d4ce2be-7e40-44ad-9bfc-5c2e276d3529, 11, Finished, Available, Finished)

2025-06-05 05:05:16,561 - INFO - Ingestion Configuration Loaded:
2025-06-05 05:05:16,562 - INFO -   USGS API Base URL: https://earthquake.usgs.gov/fdsnws/event/1/query
2025-06-05 05:05:16,562 - INFO -   Data Fetching Window: Last 365 days
2025-06-05 05:05:16,563 - INFO -   Minimum Magnitude: 2.5
2025-06-05 05:05:16,564 - INFO -   Ingestion Datetime (UTC): 2025-06-05T05:05:16.561439+00:00
2025-06-05 05:05:16,564 - INFO -   Bronze Target Table: bronze_usgs_earthquakes


In [10]:
# --- Cell 4: Function to Extract Earthquake Data ---

def fetch_earthquake_data(start_time: datetime, end_time: datetime, min_magnitude: float = 2.5, limit: int = 20000) -> pd.DataFrame:
    """
    Fetches earthquake data from the USGS API for a specified time range and minimum magnitude.
    This function handles API pagination to retrieve all available records up to the configured limit.

    The USGS API has a typical limit of 20,000 events per request. This function
    iteratively fetches data by incrementing the 'offset' parameter until all
    records are retrieved or no more data is available.

    Args:
        start_time (datetime): The start datetime for the data query (inclusive).
        end_time (datetime): The end datetime for the data query (inclusive).
        min_magnitude (float): The minimum magnitude of earthquakes to fetch. Defaults to 2.5.
        limit (int): The maximum number of events to fetch per API request. Defaults to 20000.

    Returns:
        pd.DataFrame: A Pandas DataFrame containing the extracted earthquake records.
                      Returns an empty DataFrame if no data is found or an error occurs.
    """
    # Parameters for the USGS API request.
    # Datetime objects are formatted to ISO 8601 string as required by the API.
    params = {
        'format': 'geojson',
        'starttime': start_time.strftime('%Y-%m-%dT%H:%M:%S'),
        'endtime': end_time.strftime('%Y-%m-%dT%H:%M:%S'),
        'minmagnitude': min_magnitude,
        'limit': limit,
        'offset': 1 # USGS API offset is 1-based
    }
    
    all_features = []
    
    logger.info(f"Initiating data fetch from USGS API for time range: {start_time.isoformat()} to {end_time.isoformat()} with min magnitude {min_magnitude}.")
    
    # Loop for pagination: Continuously fetch data until no more records are available
    # or the last page has been retrieved based on the 'limit' parameter.
    request_count = 0
    while True:
        request_count += 1
        logger.info(f"Making API request {request_count} with offset: {params['offset']}")
        try:
            # Execute HTTP GET request to the USGS API.
            # A timeout is critical for robust production systems to prevent indefinite waits.
            response = requests.get(USGS_API_BASE_URL, params=params, timeout=120) # Increased timeout to 120s for large responses
            response.raise_for_status()  # Raise an HTTPError for bad responses (4xx or 5xx)

            # Parse the JSON response body.
            data = response.json()
            
            # Extract the 'features' array, which contains the earthquake event details.
            features = data.get('features', [])
            all_features.extend(features)
            logger.info(f"Received {len(features)} features in current API response. Total features collected: {len(all_features)}.")
            
            # Pagination check:
            # The loop breaks if the number of features received is less than the requested limit,
            # indicating the last page, or if the API explicitly signals no more data (though
            # USGS API often doesn't provide a 'next' link directly for this endpoint).
            if len(features) < params['limit']:
                logger.info("Reached end of data or received fewer features than limit, stopping pagination.")
                break # No more data to fetch or last page reached
            
            # Increment the offset for the next request to fetch the subsequent page of data.
            params['offset'] += params['limit']
            
        except requests.exceptions.Timeout as e:
            logger.error(f"API request timed out after {response.request.timeout} seconds: {e}")
            return pd.DataFrame() # Return empty DataFrame to signal failure without crashing
        except requests.exceptions.RequestException as e:
            logger.error(f"Error fetching data from USGS API (RequestException): {e}")
            return pd.DataFrame() # Return empty DataFrame on HTTP/network errors
        except json.JSONDecodeError as e:
            logger.error(f"Error decoding JSON response from USGS API: {e}. Response content: {response.text[:500]}...")
            return pd.DataFrame() # Return empty DataFrame on invalid JSON
        except Exception as e:
            logger.error(f"An unexpected error occurred during API fetch: {e}", exc_info=True)
            return pd.DataFrame()

    if not all_features:
        logger.warning("No earthquake features found for the specified criteria. Returning empty DataFrame.")
        return pd.DataFrame()

    # --- Data Transformation to Pandas DataFrame ---
    # Iterates through each extracted feature and constructs a dictionary
    # mapping API fields to the desired column names (matching `earthquake_schema`).
    # This ensures consistency for downstream processing and storage.
    earthquakes_list = []
    for feature in all_features:
        properties = feature.get('properties', {})
        geometry = feature.get('geometry', {})
        # Coordinates are typically [longitude, latitude, depth_km]
        coordinates = geometry.get('coordinates', [None, None, None]) 
        
        earthquake_record = {
            'id': feature.get('id'),
            'mag': properties.get('mag'),
            'place': properties.get('place'),
            'time': properties.get('time'),
            'updated': properties.get('updated'),
            'tz': properties.get('tz'),
            'url': properties.get('url'),
            'detail': properties.get('detail'),
            'felt': properties.get('felt'),
            'cdi': properties.get('cdi'),
            'mmi': properties.get('mmi'),
            'alert': properties.get('alert'),
            'status': properties.get('status'),
            'tsunami': properties.get('tsunami', 0), # Default to 0 if not present
            'sig': properties.get('sig'),
            'net': properties.get('net'),
            'code': properties.get('code'),
            'ids': properties.get('ids'),
            'sources': properties.get('sources'),
            'types': properties.get('types'),
            'nst': properties.get('nst'),
            'dmin': properties.get('dmin'),
            'rms': properties.get('rms'),
            'gap': properties.get('gap'),
            'magType': properties.get('magType'),
            'type': properties.get('type'),
            'title': properties.get('title'),
            'longitude': coordinates[0] if len(coordinates) > 0 else None,
            'latitude': coordinates[1] if len(coordinates) > 1 else None,
            'depth': coordinates[2] if len(coordinates) > 2 else None,
            'ingestion_timestamp_utc': INGESTION_DATETIME # Add ingestion timestamp for data lineage
        }
        earthquakes_list.append(earthquake_record)
    
    # Create Pandas DataFrame from the list of dictionaries.
    df = pd.DataFrame(earthquakes_list)
    logger.info(f"Successfully extracted and prepared {len(df)} earthquake records as a Pandas DataFrame.")
    return df

StatementMeta(, 2d4ce2be-7e40-44ad-9bfc-5c2e276d3529, 12, Finished, Available, Finished)

In [11]:
# --- Cell 5: Execute Data Extraction ---

# This cell orchestrates the data extraction by defining the time window
# and calling the `fetch_earthquake_data` function.

# Determine the end time for data retrieval. Using `datetime.now(timezone.utc)`
# ensures consistency with the `INGESTION_DATETIME` and avoids timezone-related issues.
end_datetime_utc = datetime.now(timezone.utc)

# Calculate the start time based on the configured `DAYS_TO_FETCH` and the `end_datetime_utc`.
start_datetime_utc = end_datetime_utc - timedelta(days=DAYS_TO_FETCH)

# Log the time range for which data will be fetched. This is useful for auditing
# and understanding the scope of the current ingestion run.
logger.info(f"Initiating earthquake data fetch for the period: {start_datetime_utc.isoformat()} to {end_datetime_utc.isoformat()} (UTC).")

# Call the data fetching function with the defined parameters.
# The result is stored in a Pandas DataFrame.
df_earthquakes_pd = fetch_earthquake_data(start_datetime_utc, end_datetime_utc, MIN_MAGNITUDE)

# Validate the outcome of the data extraction.
if not df_earthquakes_pd.empty:
    # Log the successful retrieval and show a sample of the data.
    logger.info(f"Successfully fetched {len(df_earthquakes_pd)} earthquake records.")
    # For large datasets, logging the head() might produce extensive output.
    # Consider using logger.debug or only logging the count in production.
    logger.info(f"First 5 records of the fetched data:\n{df_earthquakes_pd.head().to_string()}")
else:
    # Log a warning if no data was fetched. This indicates a potential issue
    # with the API, parameters, or data availability.
    logger.warning("No earthquake data was fetched for the specified criteria. The DataFrame is empty.")
    # In a production pipeline, depending on criticality, you might want to:
    # 1. Raise an exception to fail the job if no data is an error condition.
    # 2. Proceed with an empty DataFrame if downstream steps can handle it gracefully.
    # For this script, we log and proceed, assuming subsequent steps will handle the empty DataFrame.

StatementMeta(, 2d4ce2be-7e40-44ad-9bfc-5c2e276d3529, 13, Finished, Available, Finished)

2025-06-05 05:05:45,963 - INFO - Initiating earthquake data fetch for the period: 2024-06-05T05:05:45.963554+00:00 to 2025-06-05T05:05:45.963554+00:00 (UTC).
2025-06-05 05:05:45,964 - INFO - Initiating data fetch from USGS API for time range: 2024-06-05T05:05:45.963554+00:00 to 2025-06-05T05:05:45.963554+00:00 with min magnitude 2.5.
2025-06-05 05:05:45,964 - INFO - Making API request 1 with offset: 1
2025-06-05 05:05:50,076 - INFO - Received 20000 features in current API response. Total features collected: 20000.
2025-06-05 05:05:50,077 - INFO - Making API request 2 with offset: 20001
2025-06-05 05:05:53,679 - INFO - Received 4624 features in current API response. Total features collected: 24624.
2025-06-05 05:05:53,679 - INFO - Reached end of data or received fewer features than limit, stopping pagination.
2025-06-05 05:05:53,868 - INFO - Successfully extracted and prepared 24624 earthquake records as a Pandas DataFrame.
2025-06-05 05:05:53,907 - INFO - Successfully fetched 24624 ear

In [13]:
#--- Cell 6: Convert Pandas DataFrame to Spark DataFrame and Save to Bronze Layer ---

# This cell transforms the fetched Pandas DataFrame into a Spark DataFrame
# and persists it to the Bronze layer of the Lakehouse, which serves as the
# raw, immutable landing zone for all ingested data.

if not df_earthquakes_pd.empty:
    logger.info(f"Converting Pandas DataFrame with {len(df_earthquakes_pd)} records to Spark DataFrame.")

    try:
        # In Azure Fabric notebooks, the 'spark' session is typically pre-initialized and available globally.
        # It's good practice to verify its availability.
        if 'spark' not in globals() or not isinstance(spark, SparkSession):
            logger.error("SparkSession 'spark' is not initialized. Attempting to get or create a new one.")
            spark = SparkSession.builder.appName("USGSEarthquakeIngestion").getOrCreate()
        
        # Create the Spark DataFrame using the explicitly defined schema.
        # This is a critical step for production environments as it ensures data types
        # are correctly interpreted and maintained, preventing potential schema inference issues
        # that could lead to data corruption or pipeline failures downstream.
        df_earthquakes_spark = spark.createDataFrame(df_earthquakes_pd, schema=earthquake_schema)
        logger.info(f"Successfully converted Pandas DataFrame to Spark DataFrame.")
        logger.info("Spark DataFrame Schema:")
        df_earthquakes_spark.printSchema() # Log the Spark DataFrame schema for verification

        # Write the Spark DataFrame to a Delta table in the Bronze layer.
        # This creates a managed table within the Fabric Lakehouse environment,
        # providing ACID properties, schema enforcement/evolution, and time travel capabilities.
        #
        # `format("delta")`: Specifies Delta Lake format for robust data storage.
        # `mode("overwrite")`: Replaces the entire table with the new data. For initial/full loads.
        #                      Consider `append` or `merge` for incremental update strategies.
        # `option("overwriteSchema", "true")`: Allows the target Delta table's schema to adapt
        #                                       to the incoming data's schema. Use with care in
        #                                       production for Silver/Gold layers; for Bronze, it
        #                                       can be acceptable to capture source schema evolution.
        # `saveAsTable(BRONZE_TABLE_NAME)`: Registers the data as a named table in the Lakehouse.
        logger.info(f"Writing {df_earthquakes_spark.count()} records to Bronze table: {BRONZE_TABLE_NAME} using 'overwrite' mode.")
        df_earthquakes_spark.write \
                            .format("delta") \
                            .mode("overwrite") \
                            .option("overwriteSchema", "true") \
                            .saveAsTable(BRONZE_TABLE_NAME)
        
        logger.info(f"Data successfully persisted to Bronze table: {BRONZE_TABLE_NAME}.")
        
        # Optional: Also save to raw files path if a file-based copy is desired for specific use cases.
        # This provides a redundant copy or allows access to data outside of the Delta table abstraction.
        # For long-term archiving, consider partitioning the file path (e.g., by date).
        # bronze_file_path = f"Files/bronze/usgs_earthquakes/{INGESTION_DATETIME.strftime('%Y/%m/%d')}/data_{INGESTION_DATETIME.strftime('%H%M%S')}.parquet"
        # logger.info(f"Also saving raw data to Parquet files at: {bronze_file_path}")
        # df_earthquakes_spark.write.format("parquet").mode("overwrite").save(bronze_file_path)
        # logger.info(f"Raw data successfully saved to Parquet files.")

    except Exception as e:
        logger.error(f"An error occurred during Spark DataFrame conversion or saving to Bronze layer: {e}", exc_info=True)
        # In a robust production pipeline, you might consider:
        # - Notifying operators (e.g., via email, Slack)
        # - Raising a distinct exception to halt the pipeline if this step is critical
        # - Implementing retry mechanisms
else:
    logger.warning("Skipping Spark DataFrame processing and saving to Bronze layer as no data was fetched in the previous step.")

StatementMeta(, 2d4ce2be-7e40-44ad-9bfc-5c2e276d3529, 15, Finished, Available, Finished)

2025-06-05 05:06:19,678 - INFO - Converting Pandas DataFrame with 24624 records to Spark DataFrame.
2025-06-05 05:06:24,910 - INFO - Writing 24624 records to Bronze table: bronze_usgs_earthquakes using 'overwrite' mode.
2025-06-05 05:07:09,845 - INFO - Data successfully persisted to Bronze table: bronze_usgs_earthquakes.


root
 |-- id: string (nullable = true)
 |-- mag: double (nullable = true)
 |-- place: string (nullable = true)
 |-- time: long (nullable = true)
 |-- updated: long (nullable = true)
 |-- tz: string (nullable = true)
 |-- url: string (nullable = true)
 |-- detail: string (nullable = true)
 |-- felt: long (nullable = true)
 |-- cdi: double (nullable = true)
 |-- mmi: double (nullable = true)
 |-- alert: string (nullable = true)
 |-- status: string (nullable = true)
 |-- tsunami: long (nullable = true)
 |-- sig: long (nullable = true)
 |-- net: string (nullable = true)
 |-- code: string (nullable = true)
 |-- ids: string (nullable = true)
 |-- sources: string (nullable = true)
 |-- types: string (nullable = true)
 |-- nst: long (nullable = true)
 |-- dmin: double (nullable = true)
 |-- rms: double (nullable = true)
 |-- gap: double (nullable = true)
 |-- magType: string (nullable = true)
 |-- type: string (nullable = true)
 |-- title: string (nullable = true)
 |-- longitude: double (nulla

In [14]:
# --- Cell 7: Display Sample from Bronze Table (Optional Verification) ---

# This cell performs a quick verification by querying the newly created
# or updated Bronze Delta table and displaying a sample of its contents.
# This helps confirm that the data was written correctly and is accessible
# within the Lakehouse environment.

if not df_earthquakes_pd.empty: # Only proceed if data was actually ingested
    try:
        # Ensure SparkSession is available to query the table.
        # This check is crucial if this cell were to be run in isolation
        # or after a Spark session might have expired/been reset.
        if 'spark' not in globals() or not isinstance(spark, SparkSession):
            logger.error("SparkSession 'spark' is not initialized for table verification. Attempting to get or create one.")
            spark = SparkSession.builder.appName("USGSEarthquakeVerification").getOrCreate()
            
        logger.info(f"Displaying a sample of 5 records from the Bronze table '{BRONZE_TABLE_NAME}' for verification.")
        # Retrieve the table as a Spark DataFrame and show its first 5 rows.
        # `truncate=False` ensures that column values are not truncated in the output,
        # providing a full view of the data.
        spark.table(BRONZE_TABLE_NAME).show(5, truncate=False)
        
        # Optionally, log the schema and total count for further verification.
        logger.info(f"Bronze table '{BRONZE_TABLE_NAME}' schema:")
        spark.table(BRONZE_TABLE_NAME).printSchema()
        logger.info(f"Total records in Bronze table '{BRONZE_TABLE_NAME}': {spark.table(BRONZE_TABLE_NAME).count()}")

    except Exception as e:
        logger.error(f"An error occurred while trying to read and display data from Bronze table '{BRONZE_TABLE_NAME}': {e}", exc_info=True)
else:
    logger.info("Skipping Bronze table display as no data was processed and written to the table.")

StatementMeta(, 2d4ce2be-7e40-44ad-9bfc-5c2e276d3529, 16, Finished, Available, Finished)

2025-06-05 05:07:32,768 - INFO - Displaying a sample of 5 records from the Bronze table 'bronze_usgs_earthquakes' for verification.
2025-06-05 05:07:35,955 - INFO - Bronze table 'bronze_usgs_earthquakes' schema:
2025-06-05 05:07:36,979 - INFO - Total records in Bronze table 'bronze_usgs_earthquakes': 24624


StatementMeta(, 2d4ce2be-7e40-44ad-9bfc-5c2e276d3529, 17, Finished, Available, Finished)

+----------+---+------------------------------+-------------+-------------+----+------------------------------------------------------------+----------------------------------------------------------------------------------+----+----+-----+-----+--------+-------+---+---+--------+------------------------+-------+---------------------------------+---+-----+----+-----+-------+----------------+----------------------------------------------------+---------+--------+-------+--------------------------+
|id        |mag|place                         |time         |updated      |tz  |url                                                         |detail                                                                            |felt|cdi |mmi  |alert|status  |tsunami|sig|net|code    |ids                     |sources|types                            |nst|dmin |rms |gap  |magType|type            |title                                               |longitude|latitude|depth  |ingestion_timestamp_utc   